In [11]:
import pandas as pd
import numpy as np
import os

bronze_path = "../data/bronze/"
silver_path = "../data/silver/"
os.makedirs(silver_path, exist_ok=True)

# Cargar los dataframes principales
df_clientes = pd.read_parquet(f"{bronze_path}clientes.parquet")
df_estado = pd.read_parquet(f"{bronze_path}cliente_estado_mensual.parquet")
df_prod_mensual = pd.read_parquet(f"{bronze_path}cliente_producto_mensual.parquet")
df_provincias = pd.read_parquet(f"{bronze_path}provincias.parquet")
df_segmentos = pd.read_parquet(f"{bronze_path}segmentos.parquet")
df_productos = pd.read_parquet(f"{bronze_path}productos.parquet")

print(" Todos los archivos de la capa Bronze han sido cargados exitosamente.")

 Todos los archivos de la capa Bronze han sido cargados exitosamente.


In [12]:
# Limpieza
# 1. Filtrar ids no nulos y eliminar duplicados
df_clientes = df_clientes.dropna(subset=['id_cliente'])
df_clientes = df_clientes.drop_duplicates(subset=['id_cliente'])

# 2. Eliminar columna con exceso de nulos descubierta en el análisis anterior
if 'conyuemp' in df_clientes.columns:
    df_clientes = df_clientes.drop(columns=['conyuemp'])

# 3. Corregir tipos de datos (Fechas)
df_clientes['fecha_alta'] = pd.to_datetime(df_clientes['fecha_alta'], errors='coerce')

# 4. Manejar ese único registro con nulos que vimos en el paso 1
df_clientes['sexo'] = df_clientes['sexo'].fillna(df_clientes['sexo'].mode()[0])
df_clientes['ind_empleado'] = df_clientes['ind_empleado'].fillna('N')
df_clientes['pais_residencia'] = df_clientes['pais_residencia'].fillna('EC')

print(f"🔹 Clientes procesados. Dimensiones finales: {df_clientes.shape}")

🔹 Clientes procesados. Dimensiones finales: (2000, 12)


In [13]:
# 1. Convertir fecha de corte
df_estado['fecha_corte'] = pd.to_datetime(df_estado['fecha_corte'], errors='coerce')

# 2. Imputar nulos en Edad y Renta usando la mediana
if 'edad' in df_estado.columns:
    df_estado['edad'] = df_estado['edad'].fillna(df_estado['edad'].median())

if 'renta' in df_estado.columns:
    df_estado['renta'] = df_estado['renta'].fillna(df_estado['renta'].median())

print(f"🔹 Estado mensual procesado. Dimensiones: {df_estado.shape}")

🔹 Estado mensual procesado. Dimensiones: (2000, 19)


In [14]:
# 1. Asegurar que estado_producto sea numérico
df_prod_mensual['estado_producto'] = pd.to_numeric(df_prod_mensual['estado_producto'], errors='coerce').fillna(0).astype(int)

# 2. Validar que solo existan valores 0 o 1
df_prod_mensual = df_prod_mensual[df_prod_mensual['estado_producto'].isin([0, 1])]

# 3. Eliminar duplicados por la llave compuesta
df_prod_mensual = df_prod_mensual.drop_duplicates(subset=['id_cliente', 'fecha_corte', 'id_producto'])

# Convertir fecha_corte a datetime
df_prod_mensual['fecha_corte'] = pd.to_datetime(df_prod_mensual['fecha_corte'], errors='coerce')

print(f"🔹 Producto mensual procesado. Dimensiones: {df_prod_mensual.shape}")

🔹 Producto mensual procesado. Dimensiones: (16000, 8)


In [15]:
# 1. Ordenar cronológicamente por cliente y producto
df_prod_mensual = df_prod_mensual.sort_values(by=['id_cliente', 'id_producto', 'fecha_corte'])

# 2. Obtener el estado del mes anterior para el mismo cliente y mismo producto
df_prod_mensual['estado_mes_anterior'] = df_prod_mensual.groupby(['id_cliente', 'id_producto'])['estado_producto'].shift(1)

# 3. Definir un ALTA: Mes anterior era 0 (o NaN) y mes actual es 1
es_alta = ((df_prod_mensual['estado_mes_anterior'].fillna(0) == 0) & (df_prod_mensual['estado_producto'] == 1))

# 4. Crear el dataframe de altas filtrando solo esos registros
df_altas = df_prod_mensual[es_alta].copy()

# Limpiar columnas técnicas temporales
df_altas = df_altas.drop(columns=['estado_mes_anterior'])
df_prod_mensual = df_prod_mensual.drop(columns=['estado_mes_anterior'])

print("¡La tabla 'cliente_producto_alta' ha sido generada con éxito de forma analítica!")
print(f"Se detectaron un total de {len(df_altas)} eventos de adquisición de nuevos productos.")

¡La tabla 'cliente_producto_alta' ha sido generada con éxito de forma analítica!
Se detectaron un total de 1053 eventos de adquisición de nuevos productos.


In [16]:
# Guardar los archivos limpios
df_clientes.to_parquet(f"{silver_path}clientes.parquet", index=False)
df_estado.to_parquet(f"{silver_path}cliente_estado_mensual.parquet", index=False)
df_prod_mensual.to_parquet(f"{silver_path}cliente_producto_mensual.parquet", index=False)
df_altas.to_parquet(f"{silver_path}cliente_producto_alta.parquet", index=False) # Guardamos la que creamos
df_provincias.to_parquet(f"{silver_path}provincias.parquet", index=False)
df_segmentos.to_parquet(f"{silver_path}segmentos.parquet", index=False)
df_productos.to_parquet(f"{silver_path}productos.parquet", index=False)

print("¡Se ha completado la Capa Silver y se guarda en formato Parquet!")

¡Se ha completado la Capa Silver y se guarda en formato Parquet!
